# PA 766 - Module 08
## Text Clustering and Topic Modeling with BERTopic

**Student notebook**

This activity discovers themes in **1,398 HuffPost articles from 2022**. Its centerpiece follows the *BERTopic Best Practices* “Visualize Topics” section closely: we will use topic-word bars, a document map, an intertopic distance map, and a topic hierarchy, then test what those pictures suggest by returning to the articles.

The activity uses only the article text and descriptive metadata. No publisher-assigned category column is needed.

## Learning objectives

By the end of class, students should be able to:

1. explain the practical BERTopic pipeline from document embeddings to topic representations;
2. interpret topics using both c-TF-IDF words and representative articles;
3. interpret document groups in a two-dimensional projection;
4. read an intertopic distance map and a topic hierarchy without treating either as ground truth; and
5. explain why HDBSCAN marks some documents as outliers.

## 0. Setup

Run the install cell once in Colab. If Colab asks to restart the runtime, restart it and continue with the import cell.

In [1]:
%pip install -q "bertopic==0.17.4" "sentence-transformers>=3,<7" "pandas>=2.2,<3" "plotly>=6,<8"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 56.9 MB/s eta 0:00:00


In [2]:
from pathlib import Path
import os
import random

import numpy as np
import pandas as pd
import torch
from bertopic import BERTopic
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP

pd.set_option("display.max_colwidth", 120)

## 1. Document collection

The source file is the [News Category Dataset](https://www.kaggle.com/datasets/rmisra/news-category-dataset), a collection of HuffPost articles. The prepared CSV keeps the most recent year available in the local source: **2022**.

Each modeling document combines the headline and short description. The CSV also preserves the date, author, and link so we can return from a cluster to the source context.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
#DATA_PATH = Path("YOUR PATH")
DATA_PATH = Path("/content/drive/MyDrive/a-ncsu-courses/PA766-2026/Data/HuffPost Articles/news_category_2022.csv")
news = pd.read_csv(DATA_PATH, parse_dates=["date"])
news

,document_id,date,year,headline,short_description,authors,link,text
0,news-2022-0001,2022-09-23,2022,23 Of The Funniest Tweets About Cats And Dogs This Week (Sept. 17-23),"""Until you have a dog you don't understand what could be eaten.""",Elyse Wanshel,https://www.huffpost.com/entry/funniest-tweets-cats-dogs-september-17-23_n_632de332e4b0695c1d81dc02,"23 Of The Funniest Tweets About Cats And Dogs This Week (Sept. 17-23). ""Until you have a dog you don't understand wh..."
1,news-2022-0002,2022-09-23,2022,"American Airlines Flyer Charged, Banned For Life After Punching Flight Attendant On Video","He was subdued by passengers and crew when he fled to the back of the aircraft after the confrontation, according to...",Mary Papenfuss,https://www.huffpost.com/entry/american-airlines-passenger-banned-flight-attendant-punch-justice-department_n_632e25...,"American Airlines Flyer Charged, Banned For Life After Punching Flight Attendant On Video. He was subdued by passeng..."
2,news-2022-0003,2022-09-23,2022,Over 4 Million Americans Roll Up Sleeves For Omicron-Targeted COVID Boosters,Health experts said it is too early to predict whether demand would match up with the 171 million doses of the new b...,"Carla K. Johnson, AP",https://www.huffpost.com/entry/covid-boosters-uptake-us_n_632d719ee4b087fae6feaac9,Over 4 Million Americans Roll Up Sleeves For Omicron-Targeted COVID Boosters. Health experts said it is too early to...
3,news-2022-0004,2022-09-23,2022,The Funniest Tweets From Parents This Week (Sept. 17-23),"""Accidentally put grown-up toothpaste on my toddler’s toothbrush and he screamed like I was cleaning his teeth with ...",Caroline Bologna,https://www.huffpost.com/entry/funniest-parenting-tweets_l_632d7d15e4b0d12b5403e479,"The Funniest Tweets From Parents This Week (Sept. 17-23). ""Accidentally put grown-up toothpaste on my toddler’s toot..."
4,news-2022-0005,2022-09-22,2022,Cleaner Was Dead In Belk Bathroom For 4 Days Before Body Found: Police,The 63-year-old woman was seen working at the South Carolina store on Thursday. She was found dead Monday after her ...,NaN,https://www.huffpost.com/entry/belk-worker-found-dead-columbiana-centre-bathroom_n_632c5f8ce4b0572027b0251d,Cleaner Was Dead In Belk Bathroom For 4 Days Before Body Found: Police. The 63-year-old woman was seen working at th...
...,...,...,...,...,...,...,...,...
1393,news-2022-1394,2022-01-01,2022,Austrian Holocaust Survivor 'Mrs. Gertrude' Dies At 94,Gertrude Pressburger became famous during Austria's 2016 presidential campaign after warning of the dangers of the f...,"Kirsten Grieshaber, AP",https://www.huffpost.com/entry/ap-eu-austria-obit-gertrude-pressburger_n_61d0aaa6e4b0bcd219540b75,Austrian Holocaust Survivor 'Mrs. Gertrude' Dies At 94. Gertrude Pressburger became famous during Austria's 2016 pre...
1394,news-2022-1395,2022-01-01,2022,New York City Rings In 2022 With Times Square Ball Drop Amid Smaller Crowd And COVID,"The city’s incoming mayor, Eric Adams, took his oath of office on the scene shortly afterward.","By BOBBY CAINA CALVAN, AP",https://www.huffpost.com/entry/times-square-ball-drop-eric-adams-oath_n_61cfe456e4b0c7d8b8a4586e,"New York City Rings In 2022 With Times Square Ball Drop Amid Smaller Crowd And COVID. The city’s incoming mayor, Eri..."
1395,news-2022-1396,2022-01-01,2022,Rep. Alexandria Ocasio-Cortez Rips 'Creepy Weirdo' Trump Adviser For Fixating On Boyfriend's Feet,"The congresswoman fired back at Republican Steve Cortes, turning the tables on his bizarre crack about her trip to F...",Mary Papenfuss,https://www.huffpost.com/entry/alexandria-ocasio-cortez-steve-cortes-florida-sandals-feet_n_61cf9f2de4b04b42ab7441ee,Rep. Alexandria Ocasio-Cortez Rips 'Creepy Weirdo' Trump Adviser For Fixating On Boyfriend's Feet. The congresswoman...
1396,news-2022-1397,2022-01-01,2022,Some Of The Biggest Lies Being Spread About The Jan. 6 Insurrection,The events from last Jan. 6 in Washington unfolded on live television but they've since been up for deb

In [5]:
news[["document_id", "date", "headline"]].head()

,document_id,date,headline
0,news-2022-0001,2022-09-23,23 Of The Funniest Tweets About Cats And Dogs This Week (Sept. 17-23)
1,news-2022-0002,2022-09-23,"American Airlines Flyer Charged, Banned For Life After Punching Flight Attendant On Video"
2,news-2022-0003,2022-09-23,Over 4 Million Americans Roll Up Sleeves For Omicron-Targeted COVID Boosters
3,news-2022-0004,2022-09-23,The Funniest Tweets From Parents This Week (Sept. 17-23)
4,news-2022-0005,2022-09-22,Cleaner Was Dead In Belk Bathroom For 4 Days Before Body Found: Police


In [6]:
df_test = news.sample(6, random_state=42)[
    ["headline", "short_description"]
]

In [7]:
df_test

,headline,short_description
78,'Our Hearts Are Broken': Historic Front Pages Mark The Queen's Death,Both British and international newspapers honor the passing of the U.K'.s longest-reigning monarch.
478,Herschel Walker Says There Are 52 States,There are 50.
354,Twitter Lawyer Calls Elon Musk 'Committed Enemy' As Judge Sets October Trial,Delaware Chancery Judge Kathaleen McCormick dealt the world's richest person a setback in ordering a speedy trial on...
594,"Angel Statue Beheaded At Church, $2 Million Relic Stolen",The New York City church says its stolen 18-carat gold relic was guarded by its own security system and is irreplace...
1255,"Howard Hesseman, Star Of ‘WKRP In Cincinnati,’ Dies At 81",The two-time Emmy nominee played the radio disc jockey Johnny Fever on the sitcom “WKRP in Cincinnati.”
993,Top Putin Flack Sergey Lavrov Praises Fox News,"Russia's top diplomat had some warm words for the conservative network, whose prime-time personality Tucker Carlson ..."


### Your turn: predict before modeling

Read the six sampled articles.

- What two or three themes might BERTopic discover?
- Which words in the text support each prediction?
- Which article might be difficult to place in a dense group, and why?

## 2. Build the BERTopic pipeline

![BERTopic pipeline from document embeddings to topic representations](https://maartengr.github.io/BERTopic/algorithm/default.svg)

| Step | Role in this activity |
|---|---|
| Sentence Transformer | Turns each article into a semantic embedding. |
| UMAP | Compresses embeddings so density-based clustering is workable. |
| HDBSCAN | Finds dense groups and may leave some documents as outliers. |
| c-TF-IDF | Finds words that distinguish each cluster from the other clusters. |
| Human interpretation | Checks documents and proposes a defensible topic name. |

BERTopic is modular: we explicitly supply the embedding, UMAP, HDBSCAN, and vectorizer models. BERTopic then calculates the c-TF-IDF topic representation.

In [8]:
documents = news["text"].tolist()

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.set_num_threads(1)

EMBEDDING_MODEL = "all-MiniLM-L6-v2"
EMBEDDING_REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"
embedding_model = SentenceTransformer(
    EMBEDDING_MODEL,
    revision=EMBEDDING_REVISION,
    device="cpu",
)
embeddings = embedding_model.encode(documents, show_progress_bar=True)

embeddings.shape

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/44 [00:00<?, ?it/s]

(1398, 384)

> **Discussion pause:** There are 1,398 rows in the embedding matrix, one per article. The columns are numerical features learned by the language model; they are not topic labels.

In [ ]:
umap_model = UMAP(          # Creates a model for reducing embedding dimensions.
    n_neighbors=12,         # Considers 12 nearby documents when preserving structure.
    n_components=5,         # Reduces each embedding to 5 dimensions.
    min_dist=0.0,           # Allows similar documents to be packed tightly together.
    metric="cosine",        # Measures similarity using vector direction.
    random_state=42,        # Makes results reproducible.
)

vectorizer_model = CountVectorizer(  # Converts topic documents into word-count features.
    stop_words="english",            # Removes common English words.
    ngram_range=(1, 2),              # Uses single words and two-word phrases.
    min_df=3,                        # Keeps terms appearing in at least 3 documents.
)

hdbscan_model = HDBSCAN(             # Groups similar documents into clusters.
    min_cluster_size=25,             # Requires at least 25 documents per cluster.
    min_samples=5,                   # Controls how conservative the clustering is.
    metric="euclidean",              # Uses straight-line distance after UMAP.
    cluster_selection_method="eom",  # Selects stable, persistent clusters.
    prediction_data=True,            # Saves information needed to classify new documents.
    core_dist_n_jobs=1,              # Uses one CPU job for core-distance calculations.
)

topic_model = BERTopic(                       # Creates the complete topic-modeling pipeline.
    embedding_model=embedding_model,          # Converts text into semantic embeddings.
    umap_model=umap_model,                    # Uses the configured dimension-reduction model.
    hdbscan_model=hdbscan_model,              # Uses the configured clustering model.
    vectorizer_model=vectorizer_model,        # Uses the configured word-count model.
    top_n_words=8,                            # Shows the top 8 words for each topic.
    verbose=True,                             # Displays progress while the model runs.
)

In [ ]:
topics, probabilities = topic_model.fit_transform(
    documents,   # First argument: the original text documents
    embeddings   # Second argument: the document embedding vectors
)  # Fits BERTopic and assigns a topic and probability to each document.

topic_info = topic_model.get_topic_info() #topic_model is a BERTtopic object
# Creates a summary table containing information about the discovered topics.

topic_info[["Topic", "Count", "Name", "Representation"]]
# Displays selected columns: topic ID, document count, name, and representative words.

In [ ]:
len(topic_info)

In [ ]:
model_summary = pd.Series(
    {
        "non-outlier topics": (topic_info["Topic"] != -1).sum(),
        "documents assigned to topics": (pd.Series(topics) != -1).sum(),
        "outlier documents": (pd.Series(topics) == -1).sum(),
        "outlier share": (pd.Series(topics) == -1).mean(),
    }
)
model_summary

> **Key point:** Topic numbers are identifiers, not rankings. Topic `-1` is HDBSCAN's outlier group: articles that do not sit in a sufficiently dense cluster under these settings. Keep that uncertainty visible.

## 3. Give the plots readable labels

The model's top words provide compact display labels. These labels improve the figures but do not replace interpretation or validation.

In [ ]:
readable_topic_labels = topic_model.generate_topic_labels(
    nr_words=3,          # Uses the top 3 words in each label.
    topic_prefix=True,   # Includes the topic number in each label.
    separator=" · ",     # Separates label elements with a middle dot.
)

readable_topic_labels[0] = "Outliers"
# Renames the outlier topic, which is topic -1.

topic_model.set_topic_labels(readable_topic_labels)
# Applies the custom labels to the topic model.

topic_model.get_topic_info()[["Topic", "Count", "CustomName"]]
# Displays each topic’s ID, document count, and custom label.

## 4. Interpret topics with words articles -- c-TF-IDF scores

The word bars show c-TF-IDF scores: terms that are comparatively distinctive for a topic. The document map then shows how articles are arranged in a separate two-dimensional projection. A plausible topic name should still survive a reading of representative articles.

In [ ]:
top_topic_ids = topic_info.loc[topic_info["Topic"] != -1, "Topic"].head(8).tolist()

topic_model.visualize_barchart(
    topics=top_topic_ids,
    n_words=6,
    custom_labels=True,
    width=280,
    height=280,
    title="<b>Distinctive words in the largest topics</b>",
)

In [ ]:
topic_model

### Your turn: revise one custom topic label and then create a new bar chart

Use the word bars as an initial clue. In the next cell, replace the empty dictionary `{}` with one mapping in the form `{topic_number: "your proposed label"}`. You can revise it again after reading representative articles.

In [ ]:
# Students: Revise existing custom name
topic_model.set_topic_labels(
    {##: "REVISED NAME"}
)

topic_model.get_topic_info()[
    ["Topic", "Count", "CustomName"]
]

In [ ]:
# copy and paste the bar chart code from above ^

## 5. Visualization

### Visualize documents by topic

Each point represents one article. We use headlines as hover text and create a separate two-dimensional UMAP projection for visualization. This projection does not change the fitted BERTopic model or its topic assignments.

In [ ]:
news

In [ ]:
from sklearn.base import clone

titles = news["headline"].tolist()  # Converts headlines to a list for plot labels.

umap_2d = clone(umap_model).set_params(n_components=2) # more about `n_components`: check out https://umap-learn.readthedocs.io/en/latest/parameters.html#n-components if you're interested
# Copies all UMAP settings but changes the output to two dimensions.

reduced_embeddings = umap_2d.fit_transform(embeddings)
# Creates 2D coordinates without changing the original 5D clustering model.

In [ ]:
reduced_embeddings

In [ ]:
"""
topic_model.visualize_documents(
    titles,                                  # Uses headlines to identify documents.
    reduced_embeddings=reduced_embeddings,  # Plots the precomputed 2D coordinates.
    custom_labels=True,                      # Uses the custom topic labels.
    hide_annotations=True,                   # Hides labels drawn over the plot.
)
# this code ^ prints out harmless warning messages -- I'm just suppressing the warning below.
"""


import warnings
from pandas.errors import SettingWithCopyWarning

with warnings.catch_warnings():
    warnings.simplefilter("ignore", SettingWithCopyWarning)

    fig = topic_model.visualize_documents(
        titles,
        reduced_embeddings=reduced_embeddings,
        custom_labels=True,
        hide_annotations=True,
    )

fig

> **Interpretation caution:** Nearby points have similar positions in this two-dimensional UMAP projection, but the distances are approximate. Hover over points and return to the article text before drawing conclusions about a topic or boundary.

### topis and hierarchy

This section follows the shared *BERTopic Best Practices* notebook closely. Its two go-to topic visuals are:

- `topic_model.visualize_topics()`
- `topic_model.visualize_hierarchy()`

The same calls are used below with the readable labels created earlier.

In [ ]:
topic_model.visualize_topics(custom_labels=True)

### Read the intertopic distance map

- Circle size reflects topic size.
- Nearby circles have more similar topic representations in the two-dimensional view.
- Separated circles suggest more distinct representations.

In [ ]:
topic_model.visualize_hierarchy(custom_labels=True)

### Read the topic hierarchy

The dendrogram joins the most similar topic representations first. Branches that merge later are less similar under this representation.

## 6. Export the Results

In [ ]:
document_info = topic_model.get_document_info(documents)
# Creates a DataFrame with one row for each document.
# It combines each original document with the topic assignment saved during
# fit_transform(), including the topic number, topic name, custom label,
# top keywords, assignment probability, and whether the document is a
# representative example of its topic.
# The documents must be in the same order as those used to train the model.

results = news.copy()  # Copies the original dataset.
results["topic"] = document_info["Topic"]  # Adds each document’s assigned topic ID.
results["topic_label"] = document_info["CustomName"]  # Adds its readable topic label.
results["topic_keywords"] = document_info["Top_n_words"]  # Adds its topic’s main keywords.
results["membership_strength"] = document_info["Probability"]  # Adds HDBSCAN membership strength.
results["representative_document"] = document_info["Representative_document"]  # Marks representative examples.

results[["headline", "topic", "topic_label"]].head()  # Displays the first five selected results.

results.topic.value_counts()

In [ ]:
results

### Detailed `c-TF-IDF score` values for a selected topic

The **c-TF-IDF score** measures how important a word or phrase is to a
particular topic.

A high score means the term:

- appears often in that topic, and
- helps distinguish that topic from the other topics.

It is an importance score—not a probability or a raw word count.

In [ ]:
TOPIC_TO_INSPECT = top_topic_ids[0]# Selects the first topic ID from the list of top topics.

topic_words = pd.DataFrame(
    topic_model.get_topic(TOPIC_TO_INSPECT),
    # Gets the topic's important words or phrases and their c-TF-IDF scores.

    columns=["word or phrase", "c-TF-IDF score"],
    # Gives the two resulting columns readable names.
)

representative_articles = results[
    (results["topic"] == TOPIC_TO_INSPECT)
    # Keeps only articles assigned to the selected topic.

    & (results["representative_document"])
    # Further limits the results to articles BERTopic selected as good
    # examples of that topic.
]

display(topic_words.head(8))
display(representative_articles[["headline", "short_description"]])

### Your turn: obtain a c-TF-IDF score for a different topic

## 7. Inspect the outlier group

Outliers are part of density-based clustering. They preserve evidence that some articles do not fit the dense groups well under the current settings.

In [ ]:
outliers = results[results["topic"] == -1]

outliers.sample(6, random_state=42)[
    ["headline", "short_description", "topic_keywords"]
]

### Your turn: interpret one outlier

Select one row.

- Is it genuinely unusual for this collection?
- Could it plausibly belong to a discovered topic?
- What evidence would justify leaving it unassigned?

Do not force assignment merely to make the output look complete.